<a href="https://colab.research.google.com/github/DanylchenkoKateryna/NLP-Lab-works/blob/main/notebooks/lab4_rule_based_ie.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4 — Rule-Based Information Extraction

**Dataset:** 20 Newsgroups — alt.atheism / sci.electronics / soc.religion.christian
**Task:** A — Text Classification corpus
**IE fields:** DATE · ELEC_QTY · SCRIPTURE_REF
**Method:** Regex + dictionaries (no LLM)

| Section | Content |
|---|---|
| 0 | Setup |
| 1 | Load data |
| 2 | Run extraction |
| 3 | Evaluate on gold subset |
| 4 | Error analysis (FP) |
| 5 | Edge case regression tests |
| 6 | Save audit_summary_lab4.md |

## 0. Setup

In [16]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas"])

0

In [17]:
import os, sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    if not Path('/content/NLP-Lab-works').exists():
        os.system('git clone https://github.com/DanylchenkoKateryna/NLP-Lab-works.git')
    REPO_ROOT = Path('/content/NLP-Lab-works')
else:
    _p = Path.cwd().resolve()
    REPO_ROOT = None
    for _ in range(6):
        if (_p / 'src' / 'ie_rules.py').exists():
            REPO_ROOT = _p
            break
        _p = _p.parent
    if REPO_ROOT is None:
        raise FileNotFoundError(f'Cannot locate repo root from {Path.cwd()}.')

os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / 'src'))
print('Repo root:', REPO_ROOT)

Repo root: /content/NLP-Lab-works


In [18]:
import json, re
from collections import Counter, defaultdict
from datetime import date

import pandas as pd

from ie_rules import extract_all, extract_dates, extract_elec_qty, extract_scripture_refs

pd.set_option('display.max_colwidth', 120)
print('Imports OK')

Imports OK


## 1. Load Data

Working on `data/sample/sample_processed_v2.csv` (100 rows) for quick iteration.
Full `data/processed_v2/processed_v2.csv` is used for gold evaluation but not committed to the repo.

In [19]:
sample_path = Path('data/sample/sample_processed_v2.csv')
df = pd.read_csv(sample_path)
df['text_v2'] = df['text_v2'].fillna('')
print(f'Loaded {len(df):,} rows')
df[['id', 'category', 'text_v2']].head(3)

Loaded 100 rows


,id,category,text_v2
0,0,alt.atheism,"Oops, sorry, my words, not the words of the Qur'an.\n\nNote that ""(the celestial bodies)"" in the above verse is an\n..."
1,1,soc.religion.christian,"Though there is a command in the law not to heed to one who prophecies\nfalsely, it is still possible for the one wh..."
2,2,soc.religion.christian,"I haven't followed whatever discussion there may have been on these\npeople, but I feel that C. S. Lewis is an excel..."


## 2. Run Extraction

### 2.1 Smoke test on representative sentences

In [20]:
examples = [
    "valid until 16 may 1993",
    "The oscillator runs at 32kHz with 3.7V supply and 150uA drain.",
    "See Matthew 25:31-46 and Revelation 14:9-12.",
    "1 Corinthians 11.26 — as often as you drink it.",
    "Signal went up to +14 dB at line level.",
    "Ex 33:19 shows God has mercy on whom He will.",
]

for txt in examples:
    ents = extract_all(txt)
    total = sum(len(v) for v in ents.values())
    print(f'TEXT: {txt[:75]}')
    for ft, items in ents.items():
        for it in items:
            print(f'  [{ft}] raw={it["raw"]!r}  value={it["value"]}')
    if total == 0:
        print('  (no entities found)')
    print()

TEXT: valid until 16 may 1993
  [DATE] raw='16 may 1993'  value=1993-05-16

TEXT: The oscillator runs at 32kHz with 3.7V supply and 150uA drain.
  [ELEC_QTY] raw='32kHz'  value={'numeric': 32.0, 'unit_canonical': 'kHz', 'unit_type': 'frequency'}
  [ELEC_QTY] raw='3.7V'  value={'numeric': 3.7, 'unit_canonical': 'V', 'unit_type': 'voltage'}

TEXT: See Matthew 25:31-46 and Revelation 14:9-12.
  [SCRIPTURE_REF] raw='Matthew 25:31-46'  value={'book': 'Matthew', 'chapter': 25, 'verse_start': 31, 'verse_end': 46}
  [SCRIPTURE_REF] raw='Revelation 14:9-12'  value={'book': 'Revelation', 'chapter': 14, 'verse_start': 9, 'verse_end': 12}

TEXT: 1 Corinthians 11.26 — as often as you drink it.
  [SCRIPTURE_REF] raw='1 Corinthians 11.26'  value={'book': '1 Corinthians', 'chapter': 11, 'verse_start': 26, 'verse_end': None}

TEXT: Signal went up to +14 dB at line level.
  [ELEC_QTY] raw='14 dB'  value={'numeric': 14.0, 'unit_canonical': 'dB', 'unit_type': 'level'}

TEXT: Ex 33:19 shows God has mercy o

### 2.2 Run on full sample

In [21]:
all_ents = []
for _, row in df.iterrows():
    result = extract_all(row['text_v2'])
    for ft, items in result.items():
        for it in items:
            all_ents.append({
                'text_id':    row['id'],
                'category':   row['category'],
                'field_type': ft,
                'raw':        it['raw'],
                'value':      json.dumps(it['value'], ensure_ascii=False),
                'method':     it['method'],
            })

ents_df = pd.DataFrame(all_ents)
print(f'Total extracted entities: {len(ents_df)}')
print(ents_df['field_type'].value_counts())

Total extracted entities: 39
field_type
SCRIPTURE_REF    23
ELEC_QTY         10
DATE              6
Name: count, dtype: int64


### 2.3 Distribution by newsgroup category

In [22]:
if not ents_df.empty:
    pivot = ents_df.groupby(['category', 'field_type']).size().unstack(fill_value=0)
    print(pivot)
else:
    print('No entities in sample.')

field_type              DATE  ELEC_QTY  SCRIPTURE_REF
category                                             
alt.atheism                3         0              0
sci.electronics            2        10              0
soc.religion.christian     1         0             23


## 3. Evaluate on Gold Subset

`data/sample/lab4_gold_ie.jsonl` — 120 annotated extraction instances across 64 unique texts.

**Precision** = TP / (TP + FP)
A prediction is TP if its (start_char, end_char, field_type) matches the gold annotation.

In [23]:
gold_path = Path('data/sample/lab4_gold_ie.jsonl')
gold_rows = []
with open(gold_path, encoding='utf-8') as f:
    for line in f:
        if line.strip():
            gold_rows.append(json.loads(line))

print(f'Gold entries: {len(gold_rows)}')
print(Counter(g['field_type'] for g in gold_rows))

Gold entries: 286
Counter({'SCRIPTURE_REF': 164, 'ELEC_QTY': 77, 'DATE': 45})


In [24]:
# Build gold index: text_id -> {field_type -> set of (start,end)}
gold_index = defaultdict(lambda: defaultdict(set))
for g in gold_rows:
    gold_index[g['text_id']][g['field_type']].add((g['start_char'], g['end_char']))

# Load texts for gold evaluation
full_path = Path('data/processed_v2/processed_v2.csv')
if full_path.exists():
    df_full = pd.read_csv(full_path)
    df_full['text_v2'] = df_full['text_v2'].fillna('')
else:
    df_full = df.copy()
    print('WARNING: full processed_v2.csv not found, using sample only.')

gold_text_ids = {g['text_id'] for g in gold_rows}
df_gold_texts = df_full[df_full['id'].isin(gold_text_ids)][['id', 'text_v2']]
print(f'Texts to evaluate: {len(df_gold_texts)}')

Texts to evaluate: 84


In [25]:
results = {'DATE': {'tp': 0, 'fp': 0}, 'ELEC_QTY': {'tp': 0, 'fp': 0}, 'SCRIPTURE_REF': {'tp': 0, 'fp': 0}}
fp_examples = []

for _, row in df_gold_texts.iterrows():
    tid  = int(row['id'])
    text = row['text_v2']
    ents = extract_all(text)
    for ft, items in ents.items():
        if ft not in results:
            continue
        gold_spans = gold_index[tid][ft]
        for it in items:
            span = (it['start_char'], it['end_char'])
            if span in gold_spans:
                results[ft]['tp'] += 1
            else:
                results[ft]['fp'] += 1
                if len(fp_examples) < 30:
                    fp_examples.append({
                        'text_id':    tid,
                        'field_type': ft,
                        'raw':        it['raw'],
                        'value':      str(it['value']),
                        'method':     it['method'],
                    })

prec_rows = []
for ft, counts in results.items():
    tp, fp = counts['tp'], counts['fp']
    total  = tp + fp
    prec   = tp / total if total else 0.0
    prec_rows.append({'Field Type': ft, 'TP': tp, 'FP': fp,
                      'Total extracted': total, 'Precision': f'{prec:.3f}'})

prec_df = pd.DataFrame(prec_rows)
print(prec_df.to_string(index=False))

   Field Type  TP  FP  Total extracted Precision
         DATE  39   0               39     1.000
     ELEC_QTY  77   0               77     1.000
SCRIPTURE_REF 164   0              164     1.000


In [26]:
print("=== Precision commentary ===")
for r in prec_rows:
    ft = r['Field Type']
    p  = float(r['Precision'])
    if p >= 0.90:
        note = "high precision — patterns are conservative"
    elif p >= 0.75:
        note = "moderate precision — some FP from ambiguous matches"
    else:
        note = "lower precision — review anti-rules in ie_policy.md"
    print(f"  {ft:15s}: precision={p:.3f} -> {note}")

=== Precision commentary ===
  DATE           : precision=1.000 -> high precision — patterns are conservative
  ELEC_QTY       : precision=1.000 -> high precision — patterns are conservative
  SCRIPTURE_REF  : precision=1.000 -> high precision — patterns are conservative


## 4. Error Analysis — False Positives

Up to 10 FP examples with diagnosis and the anti-rule applied.

In [27]:
fp_df = pd.DataFrame(fp_examples[:10])
if fp_df.empty:
    print("No false positives found — all extractions matched gold.")
else:
    for i, row in fp_df.iterrows():
        print(f"FP #{i+1}  [{row['field_type']}]")
        print(f"  raw:    {row['raw']!r}")
        print(f"  value:  {row['value']}")
        print(f"  method: {row['method']}")
        print()

No false positives found — all extractions matched gold.


### 4.1 Anti-rules summary

| # | FP pattern | Why wrong | Anti-rule applied |
|---|---|---|---|
| 1 | `1615A` (part number) | Letter immediately after digits | Reject if `text[start-1].isalpha()` |
| 2 | `John Smith` | Common name matches book alias | Require `:` or `.` separator for ambiguous books |
| 3 | `100%` | Percent sign, not an electrical unit | `%` absent from unit dictionary |
| 4 | `2000 years ago` | Year in phrase, no month token | MDY/DMY patterns require month name |
| 5 | `(Smith, 1985)` | Author-year citation | Date patterns require month name or ISO format |
| 6 | `0.1uF` (leading dot) | Regex requires leading digit | Pattern anchored with `\b\d+` |
| 7 | `500Ms/s` | Mega-samples/sec, not a standard unit | Not in unit dictionary → skipped |
| 8 | `Acts 1` (no verse) | Missing chapter:verse structure | Regex requires verse number after separator |
| 9 | `1960's` | Decade, not a specific date | Pattern does not match apostrophe-s suffix |
| 10 | `32 pins` | Count, not electrical measurement | `pin` not in unit dictionary |

## 5. Edge Case Regression Tests

In [28]:
ec_path = Path('tests/ie_edge_cases.jsonl')
ec_cases = []
with open(ec_path, encoding='utf-8') as f:
    for line in f:
        if line.strip():
            ec_cases.append(json.loads(line))

print(f'Loaded {len(ec_cases)} edge cases')
print(Counter(ec['field_type'] for ec in ec_cases))

Loaded 30 edge cases
Counter({'DATE': 10, 'ELEC_QTY': 10, 'SCRIPTURE_REF': 10})


In [29]:
ec_results = []
for ec in ec_cases:
    ents = extract_all(ec['raw_text'])
    ft   = ec['field_type']
    hits = ents.get(ft, [])
    is_negative = 'no extraction' in ec['expected_behavior'].lower()
    if is_negative:
        status = 'PASS' if not hits else 'FAIL'
    else:
        status = 'PASS' if hits else 'FAIL'
    ec_results.append({
        'id':        ec['id'],
        'field_type': ft,
        'category':  ec['category'],
        'expected':  ec['expected_behavior'][:55],
        'extracted': str([h['raw'] for h in hits])[:55],
        'status':    status,
    })

ec_df = pd.DataFrame(ec_results)
pass_count = (ec_df['status'] == 'PASS').sum()
fail_count = (ec_df['status'] == 'FAIL').sum()
print(f'PASS: {pass_count} / {len(ec_df)}  |  FAIL: {fail_count}')
print()
print(ec_df[['id', 'field_type', 'status', 'extracted']].to_string(index=False))

PASS: 30 / 30  |  FAIL: 0

    id    field_type status                                   extracted
IE-D01          DATE   PASS                         ['January 3, 1993']
IE-D02          DATE   PASS                             ['16 may 1993']
IE-D03          DATE   PASS                          ['September 1992']
IE-D04          DATE   PASS                                          []
IE-D05          DATE   PASS                              ['circa 1964']
IE-D06          DATE   PASS                              ['1993-04-15']
IE-D07          DATE   PASS                                          []
IE-D08          DATE   PASS                               ['4/15/1993']
IE-D09          DATE   PASS                                          []
IE-D10          DATE   PASS                                  ['Oct. 5']
IE-E01      ELEC_QTY   PASS                                   ['32kHz']
IE-E02      ELEC_QTY   PASS                                    ['3.7V']
IE-E03      ELEC_QTY   PASS          

## 6. Save audit_summary_lab4.md

In [30]:
docs_dir = Path('docs')
docs_dir.mkdir(exist_ok=True)

pass_rate = pass_count / len(ec_df) if len(ec_df) > 0 else 0.0

lines = [
    '# Audit Summary — Lab 4 Rule-Based IE',
    '',
    f'**Generated:** {date.today()}  ',
    '**Dataset:** 20 Newsgroups (alt.atheism / sci.electronics / soc.religion.christian)  ',
    '**Task:** A — Text Classification  ',
    '**Method:** Regex + dictionaries (no LLM)  ',
    '',
    '## Field Types',
    '',
    '| Field Type | Definition summary |',
    '|---|---|',
    '| DATE | Calendar dates with month names or ISO format |',
    '| ELEC_QTY | Electrical/physical quantities (V, A, Hz, Ohm, W, F, H, dB) |',
    '| SCRIPTURE_REF | Bible/Quran verse references (Book Chapter:Verse) |',
    '',
    '## Precision Results',
    '',
    '| Field Type | TP | FP | Total | Precision |',
    '|---|---|---|---|---|',
]
for r in prec_rows:
    lines.append(f"| {r['Field Type']} | {r['TP']} | {r['FP']} | {r['Total extracted']} | {r['Precision']} |")

lines += [
    '',
    '## Edge Case Tests',
    '',
    f'**Total cases:** {len(ec_df)}  ',
    f'**Pass rate:** {pass_rate:.1%}  ',
    '',
    '## Anti-Rules Applied',
    '',
    '- Reject ELEC_QTY if preceded by alpha char (part numbers)',
    '- Require colon/dot separator for ambiguous single-word book names',
    '- DATE requires month token or ISO format (no bare 4-digit years)',
    '- ELEC_QTY pattern requires leading digit (rejects `.1uF`)',
    '',
    '## Files',
    '',
    '- `src/ie_rules.py` — extractor module (regex + dicts)',
    '- `data/sample/lab4_gold_ie.jsonl` — gold subset (120 entries, 64 texts)',
    '- `tests/ie_edge_cases.jsonl` — 30 edge cases',
    '- `docs/ie_policy.md` — field-type specification',
    '- `resources/` — months_en.json, elec_units.json, bible_books.json',
]

md_path = docs_dir / 'audit_summary_lab4.md'
md_path.write_text('\n'.join(lines), encoding='utf-8')
print(f'Saved: {md_path}')
print()
print('=' * 50)
print('  Lab 4 COMPLETE')
print('=' * 50)
for r in prec_rows:
    print(f'  {r["Field Type"]:15s}: Precision={r["Precision"]}')
print(f'  Edge cases: {pass_count}/{len(ec_df)} PASS')

Saved: docs/audit_summary_lab4.md

  Lab 4 COMPLETE
  DATE           : Precision=1.000
  ELEC_QTY       : Precision=1.000
  SCRIPTURE_REF  : Precision=1.000
  Edge cases: 30/30 PASS
